In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Dataset

In [2]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

### Get Encoder Outputs

In [3]:
from scripts.image_process import process_image, image_to_tensor
from scripts.model import SwinEncoder

# Input Config
input_size = [1280, 960]
align_long_axis = False
random_padding = True

# Input Encoder
img = process_image(sample["image"], input_size, align_long_axis, random_padding)
image_tensors = image_to_tensor(img)
image_tensors = image_tensors.unsqueeze(0)
image_tensors = image_tensors.to(device)

# Encoder Config
encoder_layer = [1, 1, 1, 1]
window_size = 10
input_size = [1280, 960]

# Encoder
encoder = SwinEncoder(
    input_size=input_size,
    window_size=window_size,
    encoder_layer=encoder_layer,
).to(device)

# Inference
encoder_outputs = encoder(image_tensors)

W0930 18:45:19.241000 17933 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 18:45:19.289000 17933 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
image_tensors.shape

torch.Size([1, 3, 1280, 960])

In [5]:
encoder_outputs.shape

torch.Size([1, 40, 30, 1024])

### Tokenizer

In [6]:
from scripts.tokenizer import get_tokenizer

In [7]:
tokenizer = get_tokenizer()

In [8]:
len(tokenizer)

57568

### BART Decoder Model

In [9]:
from typing import Any, List, Optional, Union
from transformers import MBartConfig, MBartForCausalLM
from transformers.file_utils import ModelOutput
from scripts.model import BARTDecoder

In [10]:
max_position_embeddings = 768
decoder_layer = 1

In [11]:
decoder = BARTDecoder(
    max_position_embeddings=max_position_embeddings,
    decoder_layer=decoder_layer,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id
).to(device)

In [12]:
sum(p.numel() for p in decoder.parameters())

76538880

### Prepare Input For Decoder

In [13]:
from scripts.ground_truth_process import process_ground_truth, get_train_inputs

task_start_token = '<s_cord-v2>'
prompt_end_token= '<s_cord-v2>'

max_length = 768
ignore_id: int = -100
prompt_end_token_id = tokenizer.convert_tokens_to_ids(prompt_end_token)

ground_truth = sample["ground_truth"]
gt_token_sequence, _ = process_ground_truth(
    tokenizer,
    ground_truth,
    task_start_token=task_start_token,
)
processed_parse = gt_token_sequence 

input_ids, labels = get_train_inputs(tokenizer, processed_parse, prompt_end_token_id, max_length, ignore_id)
input_ids = input_ids.unsqueeze(0).to(device)
labels = labels.unsqueeze(0).to(device)
labels.shape, input_ids.shape, encoder_outputs.shape

(torch.Size([1, 768]), torch.Size([1, 768]), torch.Size([1, 40, 30, 1024]))

In [14]:
decoder_input_ids = input_ids[:, :-1]
decoder_labels = labels[:, 1:]

decoder_input_ids.shape, decoder_labels.shape

(torch.Size([1, 767]), torch.Size([1, 767]))

In [15]:
decoder_outputs = decoder(
    input_ids=decoder_input_ids,
    encoder_hidden_states=encoder_outputs.flatten(1, 2),
    labels=decoder_labels,
)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [16]:
decoder_outputs['loss']

tensor(16.2922, device='cuda:0', grad_fn=<NllLossBackward0>)